**Loading Llama3.2 model**

In [7]:
!pip install -U langchain-community mypy_extensions
!pip install -U ddgs
!pip install langchain langchain-groq langchain-classic
!pip install python-dotenv
!pip install -U sentence-transformers faiss-cpu
!pip install cohere
!pip install -U rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [8]:
import os
import numpy as np
import pandas as pd
from langchain_groq import ChatGroq
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
llm=ChatGroq( model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=1024,
)

Generating response for the prompt

In [9]:
def build_llama3_prompt(messages):
    prompt = "<|begin_of_text|>"
    for m in messages:
        prompt+= f"<|start_header_id|>{m['role']}<|end_header_id|>\n\n{m['content']}<|eot_id|>"
    prompt+= "<|start_header_id|>assistant<|end_header_id|>\n\n"
    return prompt


def generate(messages, max_new_tokens=200, do_sample=True, temperature=1, top_p=0.25):
    prompt = build_llama3_prompt(messages)
    return llm.invoke(
        prompt,
        max_tokens=max_new_tokens,
        temperature=temperature if do_sample else 0.0,
        top_p=top_p,
    )

In [10]:
messages = [
    {"role": "user", "content": "Who are the largest car manufacturers in 2023? Do they each makeEVs or not?"}
]
print(generate(messages))

content="Here's a list of the largest car manufacturers in 2023, along with their electric vehicle (EV) offerings:\n\n1. **Toyota**:\n   - EVs: Yes, Toyota offers a range of EVs, including the bZ4X, Prius Prime, and the upcoming bZ3.\n   - Market share: 10.6% (2022 global sales)\n\n2. **Volkswagen**:\n   - EVs: Yes, Volkswagen has a wide range of EVs, including the ID.4, ID.3, and the upcoming ID.5.\n   - Market share: 6.9% (2022 global sales)\n\n3. **General Motors (GM)**:\n   - EVs: Yes, GM offers a range of EVs, including the Chevrolet Bolt, Bolt EUV, and the upcoming Chevrolet Silverado EV.\n   - Market share: 6.4% (2022 global sales)\n\n4. **Ford**:\n  " additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 200, 'prompt_tokens': 65, 'total_tokens': 265, 'completion_time': 0.261668217, 'completion_tokens_details': None, 'prompt_time': 0.004326365, 'prompt_tokens_details': None, 'queue_time': 0.051888153, 'total_time': 0.265994582}, 'model_name': 'llama-3.1-8b

In [11]:
messages = [{"role": "user", "content": "Classify the text into neutral, negative or positive.\nText: I think the food was okay.\nSentiment:"}]
print(generate(messages))

content='The sentiment of the text is neutral. The word "okay" implies a lack of strong emotion or enthusiasm, indicating a neutral opinion about the food.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 68, 'total_tokens': 99, 'completion_time': 0.051316877, 'completion_tokens_details': None, 'prompt_time': 0.004407555, 'prompt_tokens_details': None, 'queue_time': 0.051717053, 'total_time': 0.055724432}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ff564-faff-7cf1-b27d-894fbba26617-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 68, 'output_tokens': 31, 'total_tokens': 99}


In [12]:
persona = "You are an expert in AI programming assistant.Help solving, writing, explaining any code to make the user's work easy.\n"
instruction= "Give a step by step answer for the user's question.If it's a coding, answer should be working code.\n"
context = "The assistance is used by the some developers.\n"
data_format = """1. Question summery.
2. Explaination
3. Code (if applicabel)
4. Conclusion\n"""
audience = "The user is beginner to intermediate programer.\n"
tone = "Friendly, professional, and concise.\n"
data = "Explain me a C program for sum of two numbers in a simple way with code."
full_prompt = persona + instruction + context + data_format + audience + tone + data
messages = [{"role": "user", "content": full_prompt}]
print(generate(messages,max_new_tokens=350))

content='**Question Summary:**\nThe user wants a C program that calculates the sum of two numbers.\n\n**Explanation:**\nIn C programming, we can use a simple program to calculate the sum of two numbers. This program will take two numbers as input from the user, add them together, and then display the result.\n\n**Code:**\n```c\n#include <stdio.h>\n\nint main() {\n    // Declare variables to store the two numbers\n    int num1, num2;\n\n    // Prompt the user to enter the first number\n    printf("Enter the first number: ");\n    scanf("%d", &num1);\n\n    // Prompt the user to enter the second number\n    printf("Enter the second number: ");\n    scanf("%d", &num2);\n\n    // Calculate the sum of the two numbers\n    int sum = num1 + num2;\n\n    // Display the result\n    printf("The sum of the two numbers is: %d\\n", sum);\n\n    return 0;\n}\n```\n\n**How the code works:**\n\n1. We include the `stdio.h` header file, which provides functions for input/output operations.\n2. We declar

**Chain-of-Thought — zero-shot version**

In [13]:
zeroshot_cot_prompt = [
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."}
]
print(generate(zeroshot_cot_prompt, max_new_tokens=200))

content="Let's break it down step by step:\n\n1. The cafeteria initially had 23 apples.\n2. They used 20 apples to make lunch, so we subtract 20 from 23:\n   23 - 20 = 3\n   Now they have 3 apples left.\n3. Then, they bought 6 more apples. To find the total number of apples they have now, we add 6 to 3:\n   3 + 6 = 9\n\nSo, the cafeteria now has 9 apples." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 80, 'total_tokens': 189, 'completion_time': 0.195137251, 'completion_tokens_details': None, 'prompt_time': 0.006121445, 'prompt_tokens_details': None, 'queue_time': 0.052181126, 'total_time': 0.201258696}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ff564-fe9f-78a0-b48c-8c9cfc9cfbfe-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 80, 'output_tokens': 109, 'total_tokens':

**Tree-of-Thought**

In [14]:
zeroshot_tot_prompt = [
    {"role": "user", "content": (
        "Imagine three different experts are answering this question. "
        "All experts will write down 1 step of their thinking, then share it with the group. "
        "Then all experts will go on to the next step, etc. "
        "If any expert realizes they're wrong at any point then they leave. "
        "The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, "
        "how many apples do they have?' Make sure to discuss the results in short."
    )}
]
print(generate(zeroshot_tot_prompt, max_new_tokens=500))

content="**Expert 1: Mathemagician**\nStep 1: The cafeteria initially had 23 apples. To find the number of apples they have now, I need to consider the changes in the number of apples.\n\n**Expert 2: Logical Linda**\nStep 1: I agree with Mathemagician. The cafeteria started with 23 apples. Now, they used 20 apples, so I need to subtract 20 from 23.\n\n**Expert 3: Statistics Steve**\nStep 1: I'm going to ignore the initial number of apples and focus on the change in the number of apples. They bought 6 more apples, so I need to add 6 to the number of apples they used.\n\n**Expert 1: Mathemagician**\nStep 2: Now that we have the initial number of apples and the change, I'll subtract the apples used from the initial number: 23 - 20 = 3.\n\n**Expert 2: Logical Linda**\nStep 2: I'll continue from where I left off. After subtracting the apples used, I have 23 - 20 = 3 apples left. Then, I need to add the 6 new apples they bought: 3 + 6 = 9.\n\n**Expert 3: Statistics Steve**\nStep 2: I made a 

**Output validation**

In [15]:
one_shot_template = """Create a short character profile for an RPG game. Make sure to only use this format:
{
  "description": "A SHORT DESCRIPTION",
  "name": "THE CHARACTER'S NAME",
  "armor": "ONE PIECE OF ARMOR",
  "weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [{"role": "user", "content": one_shot_template}]
output = generate(one_shot_prompt, max_new_tokens=150)
print(output)

content='{\n  "description": "A skilled huntress from a secluded village, trained in the art of archery and survival.",\n  "name": "Eira Shadowglow",\n  "armor": "Leather Tunic and Quiver",\n  "weapon": "Longbow and Short Sword"\n}' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 108, 'total_tokens': 170, 'completion_time': 0.090392107, 'completion_tokens_details': None, 'prompt_time': 0.009782053, 'prompt_tokens_details': None, 'queue_time': 0.052351085, 'total_time': 0.10017416}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ff565-02fa-78a1-adb2-0f624bbc256d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 108, 'output_tokens': 62, 'total_tokens': 170}


**Chain:**

*Multiple chaining*

In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}.
Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
 template=template, input_variables=["summary","title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

template = """<s><|user|>
Create a story about {summary} with the title {title}. The main character is:
{character}. Only return the story and it cannot be longer than one paragraph.
<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
 template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")
llm_chain = title | character | story
llm_chain.invoke("a girl that lost her mother")


/tmp/ipykernel_683/2270005470.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


KeyboardInterrupt: 

**Memory**: *Windowed Conversation Buffer*

In [ ]:
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.chains import LLMChain
template = """<s><|user|>Current conversation:{chat_history}
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
 template=template,
 input_variables=["input_prompt", "chat_history"]
)

memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")
llm_chain = LLMChain(
 prompt=prompt,
 llm=llm,
 memory=memory
)
llm_chain.predict(input_prompt="Hi! My name is Manoj and I am 21 years old.What is 133+ 1?")
print("\n")
llm_chain.predict(input_prompt="I forgot my name. Do you know me")

2*Conversation Summary*

In [ ]:
from langchain_classic.memory import ConversationSummaryMemory

summary_prompt_template = """<s><|user|>Summarize the conversations and update
with the new lines.
Current summary:
{summary}
new lines of conversation:
{new_lines}
New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
 input_variables=["new_lines", "summary"],
 template=summary_prompt_template
)
memory = ConversationSummaryMemory(
 llm=llm,
 memory_key="chat_history",
 prompt=summary_prompt
)
llm_chain = LLMChain(
 prompt=prompt,
 llm=llm,
 memory=memory
)
llm_chain.invoke({"input_prompt": "Hi! My name is Manoj S. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

**Agent tools**:

In [ ]:
from langchain_classic.agents import load_tools, Tool,AgentExecutor,create_react_agent
from langchain_community.tools import DuckDuckGoSearchResults

react_template = """Answer the following questions as best you can. You have
access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}"""
prompt = PromptTemplate(
 template=react_template,
 input_variables=["tools", "tool_names","input", "agent_scratchpad"]
)
search = DuckDuckGoSearchResults()
search_tool =Tool(
 name="duckduck",
 description="A web search engine. Use this to as a search engine for general queries.",
 func=search.run,
)
tools=load_tools(["llm-math"],llm=llm)
tools.append(search_tool)
agent = create_react_agent(llm, tools, prompt)
agent_executor= AgentExecutor(agent=agent,tools=tools ,verbose=True, handle_parsing_errors=True,max_iterations=6,max_execution_time=90)
agent_executor.invoke({"input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD."})

**Sematic search**

In [67]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following, and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

texts = text.split('.')
texts = [t.strip(' \n') for t in texts if t.strip(' \n')]

In [68]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeds = embedder.encode(texts, convert_to_numpy=True)
print(embeds.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(15, 384)


In [69]:
import faiss

dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
print(index.is_trained)
index.add(np.float32(embeds))

True


In [70]:
def search(query, number_of_results=3):
    query_embed = embedder.encode([query], convert_to_numpy=True)
    distances, similar_item_ids = index.search(np.float32(query_embed), number_of_results)
    texts_np = np.array(texts)
    results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]], 'distance': distances[0]})
    print(f"Query: '{query}'\nNearest neighbors:")
    return results

In [71]:
query = "how precise was the science"
results = search(query)
results

Query: 'how precise was the science'
Nearest neighbors:


,texts,distance
0,It has also received praise from many astronom...,1.048788
1,Caltech theoretical physicist and 2017 Nobel l...,1.336793
2,"Since its premiere, Interstellar gained a cult...",1.565360


*Reranking*

In [72]:
import cohere
co = cohere.Client(userdata.get('Cohere'))
query = "how precise was the science"
results = co.rerank(query=query, documents=texts, top_n=3, return_documents=True)
results.results

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics'), index=12, relevance_score=0.15239799),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'), index=10, relevance_score=0.050354082),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'), index=0, relevance_score=0.0350424)]

In [73]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)
        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc

tokenized_corpus = [bm25_tokenizer(passage) for passage in texts]
bm25 = BM25Okapi(tokenized_corpus)

In [74]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print("\nTop-3 lexical search (BM25) hits")
    for hit in bm25_hits[:top_k]:
        print(f"\t{hit['score']:.3f}\t{texts[hit['corpus_id']]}")

    docs = [texts[hit['corpus_id']] for hit in bm25_hits]

    print(f"\nTop-3 hits by rank-API ({len(docs)} BM25 hits re-ranked)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)
    for hit in results.results:
        print(f"\t{hit.relevance_score:.3f}\t{hit.document.text}")

In [75]:
keyword_and_reranking_search(query="how precise was the science")

Input question: how precise was the science

Top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles

Top-3 hits by rank-API (10 BM25 hits re-ranked)
	0.152	It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
	0.050	The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
	0.035	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan


*RAG Pipeline:*

In [ ]:
def retrieve_and_rerank(query, num_candidates=10, top_k=3):
    candidates_df = search(query, number_of_results=num_candidates)
    candidate_docs = candidates_df['texts'].tolist()
    reranked = co.rerank(
        query=query,
        documents=candidate_docs,
        top_n=top_k,
        return_documents=True
    )
    return [hit.document.text for hit in reranked.results]

In [ ]:
def rag_answer(query, num_candidates=10, top_k=3, max_new_tokens=250):
    top_chunks = retrieve_and_rerank(query, num_candidates, top_k)
    context = "\n".join(top_chunks)

    rag_instruction = (
        "Answer the question using only the information in the context below. "
        "If the answer isn't in the context, say you don't know — don't make anything up.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{query}"
    )
    messages = [{"role": "user", "content": rag_instruction}]
    answer = generate(messages, max_new_tokens=max_new_tokens)
    return answer, top_chunks

In [ ]:
query = "When was the duplicate chunk bug discovered and when was it fixed??"
answer, used_chunks = rag_answer(query)

print("Chunks used as context:")
for c in used_chunks:
    print(" -", c)

print("\nAnswer:")
print(answer.content)